In [ ]:
# -----------------------------------------------------------
# 1. IMPORTS & SETUP
# -----------------------------------------------------------
import os
import glob
import numpy as np
import faiss
import pymupdf
from sentence_transformers import SentenceTransformer, CrossEncoder
from openai import OpenAI

# Set your OpenAI API key here
os.environ["OPENAI_API_KEY"] = "your-api-key"
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [16]:
# -----------------------------------------------------------
# 2. LOAD OPEN-SOURCE MODELS
# -----------------------------------------------------------
print("Loading models...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("Models loaded successfully!")


Loading models...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 7969.20it/s]


Models loaded successfully!


In [17]:
# -----------------------------------------------------------
# 3. LOCAL FOLDER PDF LOADER (PyMuPDF) & CHUNKER
# -----------------------------------------------------------
def load_pdfs_from_folder(folder_path="data"):
    """Reads and extracts text from all PDFs in the specified local folder using PyMuPDF."""
    # Ensure the directory exists
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        print(f"Created folder '{folder_path}'. Please place PDF files inside it and re-run.")
        return ""

    pdf_files = glob.glob(os.path.join(folder_path, "*.pdf"))
    if not pdf_files:
        print(f"No PDFs found in the '{folder_path}' directory.")
        return ""

    all_text = ""
    for file_path in pdf_files:
        print(f"Extracting text from: {file_path}")
        try:
            # Open the PDF using PyMuPDF
            doc = pymupdf.open(file_path)
            for page in doc:
                extracted = page.get_text()
                if extracted:
                    all_text += extracted + "\n"
            doc.close() # Free up resources
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
            
    return all_text

def chunk_text(text, chunk_size=200, overlap=50):
    """Splits text into overlapping word chunks to fit embedding model limits."""
    if not text.strip():
        return []
    
    words = text.split()
    chunks = []
    # Slide a window over the words
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

# -----------------------------------------------------------
# 4. PREPARE KNOWLEDGE BASE & FAISS INDEX
# -----------------------------------------------------------
full_pdf_text = load_pdfs_from_folder("data")
documents = chunk_text(full_pdf_text, chunk_size=250, overlap=50)

if not documents:
    print("No text could be extracted. Pipeline halted.")
else:
    print(f"\nCreated {len(documents)} document chunks from the local PDFs.")
    print("Encoding documents into vector space (this may take a moment)...")

    # Encode the document chunks into dense vectors
    document_embeddings = embedder.encode(documents, convert_to_numpy=True)

    # Create a FAISS index
    embedding_dim = document_embeddings.shape[1]
    index = faiss.IndexFlatL2(embedding_dim)

    # Add our document embeddings to the FAISS index
    index.add(document_embeddings)
    print(f"Successfully added {index.ntotal} vectors to the FAISS index.")

Extracting text from: data/llama2.pdf

Created 212 document chunks from the local PDFs.
Encoding documents into vector space (this may take a moment)...
Successfully added 212 vectors to the FAISS index.


In [18]:
# -----------------------------------------------------------
# 5. RAG PIPELINE FUNCTION
# -----------------------------------------------------------
def rag_pipeline(query, top_k_retrieval=10, top_n_rerank=3):
    """
    Executes the full RAG pipeline: Retrieval -> Re-ranking -> Generation.
    """
    if index.ntotal == 0:
        return "The FAISS index is empty. Please add PDFs to the data folder."
        
    print(f"\n--- Processing Query: '{query}' ---")
    
    # --- STAGE 1: Fast Retrieval (Bi-Encoder + FAISS) ---
    query_embedding = embedder.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, top_k_retrieval)
    
    # Handle edge case where fewer documents exist than top_k_retrieval
    valid_indices = [idx for idx in indices[0] if idx != -1]
    retrieved_docs = [documents[idx] for idx in valid_indices]
    
    if not retrieved_docs:
        return "No relevant documents found."
    print(f"\n[Stage 1] Top {len(retrieved_docs)} Retrieved Document Chunks:")
    for i, doc in enumerate(retrieved_docs):
        print(f"  {i+1}. {doc[:100]}...")

    # --- STAGE 2: Precise Re-ranking (Cross-Encoder) ---
    cross_inp = [[query, doc] for doc in retrieved_docs]
    cross_scores = cross_encoder.predict(cross_inp)
    
    sorted_indices = np.argsort(cross_scores)[::-1]
    reranked_docs = [retrieved_docs[i] for i in sorted_indices[:top_n_rerank]]
    
    print(f"\n[Stage 2] Top {len(reranked_docs)} Re-ranked Document Chunks:")
    for i, doc in enumerate(reranked_docs):
        print(f"  {i+1}. {doc[:100]}...")

    # --- STAGE 3: Generation (OpenAI LLM) ---
    context = "\n\n---\n\n".join(reranked_docs)
    
    prompt = f"""
You are a helpful assistant. Please answer the user's question based strictly on the provided context.
If the context doesn't contain the answer, say "I don't have enough information to answer that."

Context:
{context}

Question: {query}
Answer:"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful AI assistant that answers questions based on local document data."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0
    )
    
    return response.choices[0].message.content

In [20]:
# -----------------------------------------------------------
# 6. TEST THE PIPELINE
# -----------------------------------------------------------
# Replace the query below with a question relevant to the PDFs you placed in data/
if 'index' in locals() and index.ntotal > 0:
    query = "how did llama2 approach the RLHF while training its models?"
    final_answer = rag_pipeline(query)

    print("\n=== FINAL ANSWER ===")
    print(final_answer)


--- Processing Query: 'how did llama2 approach the RLHF while training its models?' ---

[Stage 1] Top 10 Retrieved Document Chunks:
  1. datasets include Meta user data. Data Freshness The pretraining data has a cutoff of September 2022,...
  2. with the pretraining of Llama 2 using publicly available online sources. Following this, we create a...
  3. README, or by opening an issue in the GitHub repository (https://github.com/facebookresearch/llama/)...
  4. to the GQA and the two variants behaved very similar (with MQA just having a slightly larger FFN dim...
  5. subset performance and the sampling mean. No other dataset (for any choice of L) appears to have ben...
  6. 57.04 0.00 13B 62.18 0.00 34B 67.20 0.02 70B 64.14 0.01 Table 14: Evaluation of fine-tuned LLMs on d...
  7. Llama, recognized for its focus on computational efficiency during inference (Touvron et al., 2023)....
  8. relatively more violations under the unqualified advice category (although still low in an absolut